## QUESTION AND ASNWER CHAT BOT

#### Let's now imagine a scenario.
#### You're working for a car company, Sunset Motors, who wants to add a chatbot to their website to answer
#### some of customer's most frequently asked questions.
#### They have asked you to create a prototype of this FAQ chatbot.
#### So to do this, the first thing we're going to need is some data about the company.
#### Here we have our Sunset Motors context and we print this out.



In [4]:
from transformers import BertForQuestionAnswering
from transformers import BertTokenizer
import torch

C:\Users\moham\anaconda3\envs\llms_course_env\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [5]:
model_name = "bert-large-uncased-whole-word-masking-finetuned-squad" # uncased measn both upeer and lower case are considered as samw

In [6]:
model = BertForQuestionAnswering.from_pretrained(model_name)

Some weights of the model checkpoint at bert-large-uncased-whole-word-masking-finetuned-squad were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [7]:
tokenizer = BertTokenizer.from_pretrained(model_name) # speiofied the model we wwant to use the tokenizer for 

In [8]:
sunset_motors_context = "Sunset Motors is a renowned automobile dealership that has been a cornerstone of the automotive industry since its establishment in 1978. Located in the picturesque town of Crestwood, nestled in the heart of California's scenic Central Valley, Sunset Motors has built a reputation for excellence, reliability, and customer satisfaction over the past four decades. Founded by visionary entrepreneur Robert Anderson, Sunset Motors began as a humble, family-owned business with a small lot of used cars. However, under Anderson's leadership and commitment to quality, it quickly evolved into a thriving dealership offering a wide range of vehicles from various manufacturers. Today, the dealership spans over 10 acres, showcasing a vast inventory of new and pre-owned cars, trucks, SUVs, and luxury vehicles. One of Sunset Motors' standout features is its dedication to sustainability. In 2010, the dealership made a landmark decision to incorporate environmentally friendly practices, including solar panels to power the facility, energy-efficient lighting, and a comprehensive recycling program. This commitment to eco-consciousness has earned Sunset Motors recognition as an industry leader in sustainable automotive retail. Sunset Motors proudly offers a diverse range of vehicles, including popular brands like Ford, Toyota, Honda, Chevrolet, and BMW, catering to a wide spectrum of tastes and preferences. In addition to its outstanding vehicle selection, Sunset Motors offers flexible financing options, allowing customers to secure affordable loans and leases with competitive interest rates."
print(sunset_motors_context)

Sunset Motors is a renowned automobile dealership that has been a cornerstone of the automotive industry since its establishment in 1978. Located in the picturesque town of Crestwood, nestled in the heart of California's scenic Central Valley, Sunset Motors has built a reputation for excellence, reliability, and customer satisfaction over the past four decades. Founded by visionary entrepreneur Robert Anderson, Sunset Motors began as a humble, family-owned business with a small lot of used cars. However, under Anderson's leadership and commitment to quality, it quickly evolved into a thriving dealership offering a wide range of vehicles from various manufacturers. Today, the dealership spans over 10 acres, showcasing a vast inventory of new and pre-owned cars, trucks, SUVs, and luxury vehicles. One of Sunset Motors' standout features is its dedication to sustainability. In 2010, the dealership made a landmark decision to incorporate environmentally friendly practices, including solar p

In [9]:
# create a function which we'll call FAQ bot, where we can feed a question.

In [13]:
def faq_bot(question):

    context = sunset_motors_context
    input_ids = tokenizer.encode(question, context) # encode the question and the context.
    tokens = tokenizer.convert_ids_to_tokens(input_ids) #And we'll take our tokens where we convert these IDs back to their token values.
    sep_idx = input_ids.index(tokenizer.sep_token_id)  #create our segment embeddings. (sentences -> the first thing we need to do is find the separator token within our list of input IDs. So from the input ids index we're looking for the tokenizer and the separation token id. )
    # Once we've got this, we can use this to compute the number of tokens in segment A and the number oftokens in segment B.

    num_seg_a = sep_idx + 1
    num_seg_b = len(input_ids) - num_seg_a

    #We can then create our list of segment IDs.
    #So we'll do zeros repeated for each token needed in segment one.
    #And ones for each token in segment two.
    #We can then feed all this into our model.
    segment_ids = [0]*num_seg_a + [1]*num_seg_b

    # So we'll take our output and then run model, and we'll convert our input IDs into a torch tensor.
    #And our segment IDs, which will also convert into a torch tensor.
    output = model(torch.tensor([input_ids]), token_type_ids = torch.tensor([segment_ids]) )

    #We then need to create the answer star and answer end positions.
    answer_start = torch.argmax(output.start_logits)
    answer_end = torch.argmax(output.end_logits)    

    # We then tell our function that if the answer end is after the answer start.
    #So essentially is the start of the response going to be tokens ahead of the end of the response?
    #Then we can print our answer.

    #And our answer is the tokens between the start position and the end position.Plus one
    if answer_end >= answer_start:
        answer = ' '.join(tokens[answer_start:answer_end +1])
    else:
        print("I'm unable to find the answer to this question. Can you please ask me another question?")

    #In our function, we also want to tidy up any responses we might get, because sometimes hash values
    #are left in just because of how the data is tokenized.

    corrected_answer = ''
    for word in answer.split():
        if word[0:2] == '##':
            corrected_answer += word[2:]
        else:
            corrected_answer += ' ' + word
    return corrected_answer

#### So we've now got our FAQ bot function.
#### Let's ask it some questions and test its knowledge

In [15]:
faq_bot("where is the dealership Located ?")

' crestwood'

In [16]:
faq_bot("what make of cars are available?")

' ford , toyota , honda , chevrolet , and bmw'

In [18]:
faq_bot("how large is the dealership?")

' 10 acres'

You can imagine just how powerful this could be if you combined it with the Lang chain framework to

bring in even more custom data.